# Real-World Image Denoising — Stage 1
Implement, train and understand **U-Net and NAFNet on real SIDD pairs** before
researching improvements. This notebook runs the repository scripts; it does not
introduce a separate training implementation or add synthetic noise.

Choose **Runtime → Change runtime type → GPU**. Your laptop does not need a GPU.
Colab GPU availability/session limits vary. See the
[official FAQ](https://research.google.com/colaboratory/faq.html).
Run cells in order. Training cells advance one chunk each time you run them.
Standard NAFNet is the published base width-32 architecture; our shared recipe
is an initial controlled experiment, not published-score reproduction.

## 1. Mount Drive and edit these paths
Upload/copy the **already downloaded SIDD Medium sRGB** folder or ZIP to Drive.
Set either `SIDD_FOLDER` or `SIDD_ZIP` below. Keep the original scene names.
Use a unique `EXPERIMENT` for each new protocol/model size/settings combination.

Use `PROTOCOL = "official"` only if you also have the original validation noisy
and GT MAT files; otherwise use the scene-disjoint `internal` protocol. Internal
scores are not official SIDD benchmark scores. Do not alter settings when resuming.

In [ ]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')

EXPERIMENT = 'stage1_standard_seed42'
MODEL_SIZE = 'standard'  # 'standard' or reduced 'laptop'; do not mix within a run
PROTOCOL = 'internal'   # 'internal' or 'official'
SIDD_FOLDER = Path('/content/drive/MyDrive/SIDD_Medium_Srgb')
SIDD_ZIP = ''           # e.g. '/content/drive/MyDrive/SIDD_Medium_Srgb.zip'; overrides folder
VAL_NOISY = Path('/content/drive/MyDrive/ValidationNoisyBlocksSrgb.mat')
VAL_CLEAN = Path('/content/drive/MyDrive/ValidationGtBlocksSrgb.mat')
STEPS_PER_CHUNK = 1000

assert re.fullmatch(r'[A-Za-z0-9_-]+', EXPERIMENT), 'Use only letters/numbers/_/- for EXPERIMENT'
assert MODEL_SIZE in ('standard', 'laptop') and PROTOCOL in ('internal', 'official')
assert isinstance(STEPS_PER_CHUNK, int) and STEPS_PER_CHUNK > 0
WORK = Path('/content/drive/MyDrive/ImageDenoising') / EXPERIMENT
LOCAL_DATA = Path('/content/sidd_source') / EXPERIMENT
PREPARED = WORK / 'prepared'
REPO = Path('/content/SOTA-Image-Denoising')
WORK.mkdir(parents=True, exist_ok=True)
print('Persistent outputs:', WORK)
print('Transient dataset copy:', LOCAL_DATA)

## 2. Get the code and inspect the runtime
The first run records a Git revision in Drive. Reconnecting uses that revision,
so a later repo update does not silently change an existing experiment. Create a
new experiment name to intentionally use updated code. Install only missing or
incompatible requirements; keep Colab's working CUDA PyTorch build when supported.

In [ ]:
import os

def run(*args):
    subprocess.run([str(arg) for arg in args], cwd=REPO, check=True)

if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/nitya-goyal/SOTA-Image-Denoising.git', str(REPO)], check=True)
revision_file = WORK / 'code_revision.txt'
if revision_file.exists():
    revision = revision_file.read_text().strip()
    assert re.fullmatch(r'[0-9a-f]{40}', revision), 'Invalid saved code revision'
    run('git', 'checkout', '--detach', revision)
else:
    revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    revision_file.write_text(revision)
os.chdir(REPO)
run(sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import torch
print('Revision:', revision)
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print('Select a GPU runtime before sustained training. CPU is available for checks only.')
print('Local disk free (GiB):', round(shutil.disk_usage('/content').free / 2**30, 1))
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(WORK / 'environment_latest.txt').write_text(freeze)

## 3. Stage the existing dataset on the runtime's local disk
Reading thousands of training patches directly from Drive is slow. Copy the
folder once per runtime, or extract its ZIP locally. This never changes the
Drive source. Allow enough local disk space for the extracted data. A completed
copy is reused in the same runtime; after a VM reset it must be staged again.

In [ ]:
import zipfile
import stat

source = Path(SIDD_ZIP) if SIDD_ZIP else SIDD_FOLDER
if not source.exists():
    raise FileNotFoundError(f'Edit the dataset path in cell 1: {source}')
stage_info = {'source': str(source), 'zip': bool(SIDD_ZIP)}
marker = LOCAL_DATA / '.stage_complete.json'
settings = {'model_size': MODEL_SIZE, 'protocol': PROTOCOL, 'seed': 42,
            'source': str(source), 'val_noisy': str(VAL_NOISY) if PROTOCOL == 'official' else None,
            'val_clean': str(VAL_CLEAN) if PROTOCOL == 'official' else None}
settings_file = WORK / 'experiment.json'
if settings_file.exists() and json.loads(settings_file.read_text()) != settings:
    raise ValueError('Experiment settings changed. Restore them or choose a new EXPERIMENT name.')
if marker.exists():
    assert json.loads(marker.read_text()) == stage_info, 'Staged source differs; use a new experiment'
    print('Using completed local dataset copy')
else:
    if LOCAL_DATA.exists():
        raise RuntimeError('Incomplete local staging folder. Remove only this transient folder manually, then rerun: ' + str(LOCAL_DATA))
    LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)
    if SIDD_ZIP:
        with zipfile.ZipFile(source) as archive:
            required = sum(member.file_size for member in archive.infolist())
            if required + 512 * 2**20 > shutil.disk_usage('/content').free:
                raise RuntimeError('Not enough local disk space to extract this archive')
            for member in archive.infolist():
                destination = (LOCAL_DATA / member.filename).resolve()
                if not destination.is_relative_to(LOCAL_DATA.resolve()) or stat.S_ISLNK(member.external_attr >> 16):
                    raise ValueError('Unsafe archive member: ' + member.filename)
            LOCAL_DATA.mkdir()
            archive.extractall(LOCAL_DATA)
    else:
        if not source.is_dir():
            raise ValueError('SIDD_FOLDER must be a directory; set SIDD_ZIP for an archive')
        required = sum(path.stat().st_size for path in source.rglob('*') if path.is_file())
        if required + 512 * 2**20 > shutil.disk_usage('/content').free:
            raise RuntimeError('Not enough local disk space to copy this dataset')
        shutil.copytree(source, LOCAL_DATA)
    marker.write_text(json.dumps(stage_info, indent=2))
settings_file.write_text(json.dumps(settings, indent=2))
print('Dataset staged at:', LOCAL_DATA)

## 4. Verify the real image pairs and prepare fixed splits
No data is generated if the dataset is missing. Original SIDD scene folders are
required. Internal splits group all camera/ISO variants by physical scene.
Official mode exports the supplied validation blocks and creates no test split.
Persistent manifests point to the same deterministic `/content` location after
reconnects; always restage the dataset before resuming.

In [ ]:
from datasets.paired_dataset import PairedDataset
from utils.runtime import file_hash

info_file = PREPARED / 'split_info.json'
if not info_file.exists():
    arguments = [sys.executable, 'prepare_sidd.py', '--source', LOCAL_DATA,
                 '--output', PREPARED, '--protocol', PROTOCOL, '--seed', '42']
    if PROTOCOL == 'official':
        if not VAL_NOISY.is_file() or not VAL_CLEAN.is_file():
            raise FileNotFoundError('Official protocol requires both original validation MAT files; edit cell 1')
        arguments += ['--val-noisy', VAL_NOISY, '--val-clean', VAL_CLEAN]
    run(*arguments)
info = json.loads(info_file.read_text())
assert info['protocol'] == PROTOCOL and info['seed'] == 42
for split, entry in info['splits'].items():
    manifest = PREPARED / f'{split}.csv'
    assert file_hash(manifest) == entry['sha256'], 'Manifest changed; do not alter an existing split'
    data = PairedDataset(manifest)
    print(split, ':', len(data), 'pairs; groups:', entry['groups'])
print('Protocol:', PROTOCOL)

## 5. Preserve the experiment configs
Both models use the same training budget, data, crops, optimizer and loss. Their
parameter counts differ. Standard NAFNet matches the original width-32 base block
configuration; laptop NAFNet is a smaller variant. These configs use a controlled
20,000-step recipe, not the original paper's full training recipe or TLC inference.

In [ ]:
config_dir = WORK / 'configs'
config_dir.mkdir(exist_ok=True)
CONFIGS = {}
for model_name in ('unet', 'nafnet'):
    original = json.loads((REPO / 'configs' / f'{model_name}_{MODEL_SIZE}.json').read_text())
    saved = config_dir / f'{model_name}_{MODEL_SIZE}.json'
    if saved.exists() and json.loads(saved.read_text()) != original:
        raise ValueError('Saved config differs; use a new experiment for intentional changes: ' + str(saved))
    saved.write_text(json.dumps(original, indent=2))
    CONFIGS[model_name] = saved
    print(model_name, ':', original)

## 6. Short sanity run for both models
This checks real-data loading and GPU execution. It does not establish denoising
quality. Validation still visits the full validation manifest and can take time.
Each smoke run saves separately from the main experiment. No unit-test fixture
images enter this experiment.

In [ ]:
def train_chunk(model_name, smoke=False):
    if not smoke and not torch.cuda.is_available():
        raise RuntimeError('Select a Colab GPU runtime before sustained training')
    output = WORK / ('smoke' if smoke else 'runs') / model_name
    checkpoint = output / 'last.pt'
    current = torch.load(checkpoint, map_location='cpu', weights_only=True)['step'] if checkpoint.exists() else 0
    config = json.loads(CONFIGS[model_name].read_text())
    target = min(2 if smoke else current + STEPS_PER_CHUNK, config['max_steps'])
    if current >= target:
        print(model_name, ': already at', current, 'steps')
        return
    arguments = [sys.executable, 'train.py', '--config', CONFIGS[model_name], '--data', PREPARED,
                 '--output', output, '--device', 'auto', '--stop-after', target]
    if checkpoint.exists():
        arguments += ['--resume', checkpoint]
    run(*arguments)
    print('Saved:', output)

train_chunk('unet', smoke=True)
train_chunk('nafnet', smoke=True)

## 7. Train U-Net — rerun this cell to advance another chunk
Check the timing from the first chunk before planning a longer session. Checkpoints
are saved every validation interval and at each requested chunk stop. Reconnecting
requires rerunning cells 1–5 and the function definition in cell 6. Existing smoke
runs are skipped; main training resumes from Drive's `last.pt`.

In [ ]:
train_chunk('unet')

## 8. Train NAFNet — rerun this cell to advance another chunk
Train both references to the same planned budget before the main comparison.
Use a new experiment with the reduced pair if standard models run out of memory.
Keep completed reference runs unchanged for future improvement experiments.

In [ ]:
train_chunk('nafnet')

## 9. Evaluate and save results to Drive
For the internal protocol, avoid inspecting the test set until model choices and
training budget are fixed; use validation for preliminary diagnostics. This cell
uses validation while either run is incomplete, and internal test only after
both runs reach their configured budgets. Official mode always uses validation.
Results are never labeled hidden-test benchmark scores. Timing includes tile
inference after warm-up, excluding data loading, transfers, metrics and image saving.

In [ ]:
from datetime import datetime, timezone

selected = []
completed = True
for model_name in ('unet', 'nafnet'):
    folder = WORK / 'runs' / model_name
    if not (folder / 'last.pt').is_file() or not (folder / 'best.pt').is_file():
        raise FileNotFoundError('Run the main training cell first: ' + model_name)
    state = torch.load(folder / 'last.pt', map_location='cpu', weights_only=True)
    configured = json.loads(CONFIGS[model_name].read_text())['max_steps']
    completed = completed and state['step'] >= configured
    print(model_name, 'completed steps:', state['step'], '/', configured)
    selected.append(folder / 'best.pt')
split = 'test' if PROTOCOL == 'internal' and completed else 'val'
label = ('internal_test' if split == 'test' else
         'official_validation' if PROTOCOL == 'official' else 'internal_validation')
if not completed:
    label = 'preliminary_' + label
    print('Training budget is incomplete. These are preliminary validation diagnostics.')
RESULTS = WORK / 'results' / (label + '_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
run(sys.executable, 'evaluate.py', '--manifest', PREPARED / f'{split}.csv',
    '--checkpoints', *selected, '--output', RESULTS, '--tile-size', '256', '--overlap', '32', '--repeats', '3')
print('Results saved:', RESULTS)
print((RESULTS / 'summary.json').read_text())

## 10. Preview and download the small results bundle
Images show **noisy | prediction | ground truth**. Keep the full Drive experiment
folder for resuming and auditing; the ZIP below includes results only, not dataset
images or large model checkpoints. Future work starts by analyzing these failures
and recent literature, then testing a specific improvement against these references.

In [ ]:
from IPython.display import display
from PIL import Image
from google.colab import files

for image_path in sorted(RESULTS.glob('*.png'))[:2]:
    display(Image.open(image_path))
archive_path = shutil.make_archive(str(RESULTS), 'zip', root_dir=RESULTS)
print('Full experiment remains in Drive:', WORK)
files.download(archive_path)